# Final Summary Figures for TeX

Notebook ini membuat artefak ringkasan akhir untuk repo myocardial-infarction-localization. Output utama disimpan ke `fig-tex/` dan dicerminkan ke `file-tex/figs/`.

Artefak utama:
- `data_distribution_ring_chart.png`
- `ecg_raw_vs_butterworth_lead_ii.png`
- `butterworth_bandpass_response.png`
- `final_macro_f1_model_comparison_bar.png`
- `final_model_comparison_summary.csv`
- `best_roc_curve.png`
- `best_pr_curve.png`
- `best_confusion_matrix.png`
- `best_training_curve.png`
- `selected_best_fold_figures.json`


In [33]:

from __future__ import annotations

import json
import math
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import butter, decimate, filtfilt, freqz


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates.append(Path('/home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization'))
    for candidate in candidates:
        if (candidate / 'dataset').exists() and (candidate / 'outputs').exists() and (candidate / 'notebook').exists():
            return candidate
    raise FileNotFoundError('Cannot locate myocardial-infarction-localization project root.')

PROJECT_ROOT = find_project_root()
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
DATASET_DIR = PROJECT_ROOT / 'dataset'
FIG_DIR = PROJECT_ROOT / 'fig-tex'
LATEX_FIG_DIR = PROJECT_ROOT / 'file-tex' / 'figs'
for path in [FIG_DIR, LATEX_FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FS = 100
RAW_PTB_DIAGNOSTIC_FS = 1000
LOWCUT = 0.5
HIGHCUT = 40.0
USED_BUTTER_ORDER = 4
BUTTER_ORDERS_TO_COMPARE = [2, 4, 6, 8]
SQRT_HALF_POWER = 1 / math.sqrt(2)

CLASS_NAMES = ['NORM', 'AMI', 'IMI', 'LMI']
CLASS_COLORS = ['#2F6FDE', '#F59E0B', '#13A376', '#D64550']
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 450,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})


def save_artifact(fig: plt.Figure, filename: str) -> Path:
    out = FIG_DIR / filename
    fig.savefig(out, bbox_inches='tight', dpi=450)
    shutil.copy2(out, LATEX_FIG_DIR / filename)
    plt.close(fig)
    return out


def copy_artifact(src: Path, filename: str) -> Path | None:
    if src is None or not Path(src).exists():
        return None
    out = FIG_DIR / filename
    shutil.copy2(src, out)
    shutil.copy2(out, LATEX_FIG_DIR / filename)
    return out


def copy_text_artifact(path: Path) -> None:
    shutil.copy2(path, LATEX_FIG_DIR / path.name)


def latest_run(root: Path) -> Path | None:
    if not root.exists():
        return None
    runs = [p for p in root.iterdir() if p.is_dir() and p.name[:8].isdigit()]
    return sorted(runs)[-1] if runs else None


def first_existing(paths) -> Path | None:
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


def find_raw_ptb_diagnostic_root() -> Path | None:
    search_bases = [PROJECT_ROOT, *PROJECT_ROOT.parents, Path.home() / 'code-program' / 'code-thesis', Path.home() / 'code-program' / 'code-thesis' / 'hibah']
    seen = set()
    for base in search_bases:
        if not base.exists() or base in seen:
            continue
        seen.add(base)
        for current_root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in {'dump_artifact', '.git', '.venv', 'outputs'}]
            if 'data_raw.npz' in files and 'meta.csv' in files:
                return Path(current_root)
    return None

print('Project root:', PROJECT_ROOT)
print('Output root:', OUTPUTS_DIR)
print('fig-tex:', FIG_DIR)
print('LaTeX mirror:', LATEX_FIG_DIR)


Project root: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization
Output root: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/outputs
fig-tex: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/fig-tex
LaTeX mirror: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/file-tex/figs


In [34]:

# 1) Butterworth bandpass response with several orders.
nyquist = 0.5 * FS
fig, ax = plt.subplots(figsize=(8.8, 4.8))
for order in BUTTER_ORDERS_TO_COMPARE:
    b, a = butter(order, [LOWCUT / nyquist, HIGHCUT / nyquist], btype='bandpass')
    w, h = freqz(b, a, worN=4096, fs=FS)
    ax.plot(
        w,
        np.abs(h),
        linewidth=2.4 if order == USED_BUTTER_ORDER else 1.35,
        alpha=1.0 if order == USED_BUTTER_ORDER else 0.70,
        label=f'Order {order}' + (' (used)' if order == USED_BUTTER_ORDER else ''),
    )
ax.axhline(SQRT_HALF_POWER, color='#7C2D12', linestyle='--', linewidth=1.15, label=r'$1/\sqrt{2}$ cutoff')
ax.axvline(LOWCUT, color='#475569', linestyle=':', linewidth=1.0)
ax.axvline(HIGHCUT, color='#475569', linestyle=':', linewidth=1.0)
ax.axvspan(LOWCUT, HIGHCUT, color='#FED7AA', alpha=0.35, label='Passband 0.5-40 Hz')
ax.set_xlim(0, 50)
ax.set_ylim(-0.03, 1.08)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude |H(f)|')
ax.set_title('Butterworth Bandpass Response for ECG Preprocessing')
ax.legend(frameon=False, ncol=2, loc='lower center', bbox_to_anchor=(0.5, -0.36))
ax.grid(True, alpha=0.25)
save_artifact(fig, 'butterworth_bandpass_response.png')
print('Saved butterworth_bandpass_response.png')


Saved butterworth_bandpass_response.png


In [35]:

# 2) Dataset class distribution from generated beat dataset.
summary_path = DATASET_DIR / 'combined_split_summary.csv'
if not summary_path.exists():
    raise FileNotFoundError(f'Missing dataset summary: {summary_path}')
summary_df = pd.read_csv(summary_path)
label_cols = ['Normal_beats', 'Anterior_beats', 'Inferior_beats', 'Lateral_beats']
plot_summary = summary_df.groupby('dataset_source', as_index=False)[label_cols].sum()
name_map = {'ptb_xl': 'PTB-XL', 'ptb_diagnostic': 'PTB Diagnostic'}
plot_summary['dataset_display'] = plot_summary['dataset_source'].map(name_map).fillna(plot_summary['dataset_source'])

fig, axes = plt.subplots(1, len(plot_summary), figsize=(11.2, 5.3))
if len(plot_summary) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, plot_summary.iterrows()):
    counts = row[label_cols].to_numpy(dtype=float)
    wedges, _ = ax.pie(
        counts,
        colors=CLASS_COLORS,
        startangle=90,
        counterclock=False,
        wedgeprops={'width': 0.38, 'edgecolor': 'white', 'linewidth': 2.0},
    )
    total = int(counts.sum())
    ax.text(0, 0.08, f'{total:,}', ha='center', va='center', fontsize=16, fontweight='bold')
    ax.text(0, -0.15, 'beats', ha='center', va='center', fontsize=9, color='#475569')
    ax.set_title(row['dataset_display'], fontsize=13, fontweight='bold')
    ax.set_aspect('equal')
    labels = [f'{name}: {int(count):,} ({count / total * 100:.1f}%)' for name, count in zip(CLASS_NAMES, counts)]
    ax.legend(wedges, labels, loc='lower center', bbox_to_anchor=(0.5, -0.26), frameon=False, fontsize=8)
fig.suptitle('Beat-Level Class Distribution of PTB-XL and PTB Diagnostic', fontsize=15, fontweight='bold')
fig.subplots_adjust(bottom=0.22, wspace=0.28)
save_artifact(fig, 'data_distribution_ring_chart.png')
plot_summary.to_csv(FIG_DIR / 'data_distribution_summary.csv', index=False)
copy_text_artifact(FIG_DIR / 'data_distribution_summary.csv')
print(plot_summary.to_string(index=False))
print('Saved data_distribution_ring_chart.png')


dataset_source  Normal_beats  Anterior_beats  Inferior_beats  Lateral_beats dataset_display
ptb_diagnostic           762             501             942            753  PTB Diagnostic
        ptb_xl        100582            6743           12300          12403          PTB-XL
Saved data_distribution_ring_chart.png


In [36]:

# 3) Raw versus Butterworth-filtered ECG Lead II from raw PTB Diagnostic if available.
raw_root = find_raw_ptb_diagnostic_root()
if raw_root is None:
    raise FileNotFoundError('Cannot locate PTB Diagnostic raw root containing data_raw.npz and meta.csv.')
raw_npz = np.load(raw_root / 'data_raw.npz')
meta = pd.read_csv(raw_root / 'meta.csv')
preferred_keys = ['patient043/s0141lre']
key = next((k for k in preferred_keys if k in raw_npz.files), raw_npz.files[0])
patient, record_id = key.split('/') if '/' in key else ('unknown', key)
raw_signal = np.asarray(raw_npz[key], dtype=np.float32)
lead_idx = LEAD_NAMES.index('II')
raw = raw_signal[:10 * RAW_PTB_DIAGNOSTIC_FS, lead_idx]
raw_fs = RAW_PTB_DIAGNOSTIC_FS
b_raw, a_raw = butter(USED_BUTTER_ORDER, [LOWCUT / (0.5 * raw_fs), HIGHCUT / (0.5 * raw_fs)], btype='bandpass')
filtered = filtfilt(b_raw, a_raw, raw)
time = np.arange(raw.shape[0]) / raw_fs

fig, axes = plt.subplots(3, 1, figsize=(11.4, 7.4), sharex=True, gridspec_kw={'height_ratios': [1.0, 1.0, 0.72]})
axes[0].plot(time, raw, color='#111827', linewidth=0.85)
axes[0].set_title(f'Raw ECG Lead II - PTB Diagnostic {patient}/{record_id}')
axes[0].set_ylabel('Amplitude')
axes[1].plot(time, filtered, color='#EA580C', linewidth=0.95)
axes[1].set_title(f'Butterworth Filtered Lead II ({LOWCUT}-{HIGHCUT} Hz, order {USED_BUTTER_ORDER})')
axes[1].set_ylabel('Amplitude')
axes[2].plot(time, raw, color='#94A3B8', linewidth=0.70, label='Raw')
axes[2].plot(time, filtered, color='#DC2626', linewidth=0.90, label='Filtered')
axes[2].set_title('Overlay')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Amplitude')
axes[2].legend(frameon=False, loc='upper right')
for ax in axes:
    ax.grid(True, alpha=0.25)
fig.suptitle('ECG Lead II Before and After Butterworth Bandpass Filtering', fontsize=15, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.96))
save_artifact(fig, 'ecg_raw_vs_butterworth_lead_ii.png')
print(f'Saved ecg_raw_vs_butterworth_lead_ii.png from {key}')


Saved ecg_raw_vs_butterworth_lead_ii.png from patient043/s0141lre


In [37]:

# 4) Merge final metrics from ALPA-Net, no-attention ablation, and baseline comparison notebooks.
MODEL_LABELS = {
    'no_pretrain': 'ALPA-Net scratch',
    'frozen_backbone': 'ALPA-Net frozen backbone',
    'partial_finetune': 'ALPA-Net partial fine-tune',
    'full_finetune': 'ALPA-Net full fine-tune',
    'cnn1d': 'CNN1D',
    'cnn_lstm': 'CNN-LSTM',
    'gru': 'GRU',
    'lstm': 'LSTM',
    'hubert_ecg': 'HuBERT-ECG',
    'ecg_fm': 'ECG-FM',
}

summary_rows = []
artifact_rows = []


def add_pretrain_run(root_name: str, display_model: str, group: str):
    run = latest_run(OUTPUTS_DIR / root_name)
    if run is None:
        return
    final_path = run / 'metrics' / 'final_metrics.csv'
    if not final_path.exists():
        return
    df = pd.read_csv(final_path)
    test = df[df['split'].astype(str).eq('test')].iloc[0]
    summary_rows.append({
        'experiment_group': group,
        'model': display_model,
        'setting': 'pretrain',
        'test_macro_f1_mean': float(test['macro_f1']),
        'test_macro_f1_std': np.nan,
        'test_weighted_f1_mean': float(test.get('weighted_f1', np.nan)),
        'test_balanced_accuracy_mean': float(test.get('balanced_accuracy', np.nan)),
        'test_macro_auc_mean': float(test.get('macro_auc', np.nan)),
        'source_run': str(run.relative_to(PROJECT_ROOT)),
    })
    artifact_rows.append({
        'model': display_model,
        'setting': 'pretrain',
        'test_macro_f1': float(test['macro_f1']),
        'roc_path': run / 'metrics' / 'test_roc_curve.png',
        'pr_path': run / 'metrics' / 'test_pr_curve.png',
        'confusion_path': run / 'metrics' / 'test_confusion_matrix.png',
        'training_curve_path': run / 'metrics' / 'training_curve.png',
        'source_run': str(run.relative_to(PROJECT_ROOT)),
    })


def add_cv_run(root_name: str, group: str, label_prefix: str):
    run = latest_run(OUTPUTS_DIR / root_name)
    if run is None:
        return
    mean_path = run / 'metrics' / 'cross_validation_mean_std_by_strategy.csv'
    detail_path = run / 'metrics' / 'detailed_fold_metrics.csv'
    if not mean_path.exists() or not detail_path.exists():
        return
    mean_df = pd.read_csv(mean_path)
    detail_df = pd.read_csv(detail_path)
    for _, row in mean_df.iterrows():
        strategy = row['strategy']
        model = f"{label_prefix} {MODEL_LABELS.get(strategy, strategy)}"
        summary_rows.append({
            'experiment_group': group,
            'model': model,
            'setting': strategy,
            'test_macro_f1_mean': float(row['test_main_macro_f1_mean']),
            'test_macro_f1_std': float(row.get('test_main_macro_f1_std', np.nan)),
            'test_weighted_f1_mean': float(row.get('test_main_weighted_f1_mean', np.nan)),
            'test_balanced_accuracy_mean': float(row.get('test_balanced_accuracy_mean', np.nan)),
            'test_macro_auc_mean': float(row.get('test_macro_auc_mean', np.nan)),
            'source_run': str(run.relative_to(PROJECT_ROOT)),
        })
    ok = detail_df[detail_df.get('status', 'OK').astype(str).eq('OK')].copy()
    if not ok.empty:
        for strategy, sub in ok.groupby('strategy'):
            best = sub.sort_values('test_main_macro_f1', ascending=False).iloc[0]
            out_dir = Path(best['output_dir'])
            if not out_dir.is_absolute():
                out_dir = PROJECT_ROOT / out_dir
            artifact_rows.append({
                'model': f"{label_prefix} {MODEL_LABELS.get(strategy, strategy)}",
                'setting': strategy,
                'test_macro_f1': float(best['test_main_macro_f1']),
                'roc_path': out_dir / 'metrics' / 'test_roc_curve.png',
                'pr_path': out_dir / 'metrics' / 'test_pr_curve.png',
                'confusion_path': out_dir / 'metrics' / 'test_confusion_matrix.png',
                'training_curve_path': out_dir / 'metrics' / 'training_curve.png',
                'source_run': str(run.relative_to(PROJECT_ROOT)),
            })


def add_single_model_comparison(root_name: str, group: str, setting: str):
    run = latest_run(OUTPUTS_DIR / root_name)
    if run is None:
        return
    metrics_path = run / 'metrics_summary.csv'
    if not metrics_path.exists():
        return
    df = pd.read_csv(metrics_path)
    if 'status' in df.columns:
        df = df[df['status'].astype(str).eq('OK')]
    for _, row in df.iterrows():
        model_key = str(row['model_name'])
        model = MODEL_LABELS.get(model_key, model_key)
        summary_rows.append({
            'experiment_group': group,
            'model': model,
            'setting': setting,
            'test_macro_f1_mean': float(row['test_macro_f1']),
            'test_macro_f1_std': np.nan,
            'test_weighted_f1_mean': float(row.get('test_weighted_f1', np.nan)),
            'test_balanced_accuracy_mean': float(row.get('test_balanced_accuracy', np.nan)),
            'test_macro_auc_mean': float(row.get('test_macro_auc', np.nan)),
            'source_run': str(run.relative_to(PROJECT_ROOT)),
        })
        out_dir = Path(row.get('output_dir', run / model_key))
        if not out_dir.is_absolute():
            out_dir = PROJECT_ROOT / out_dir
        artifact_rows.append({
            'model': model,
            'setting': setting,
            'test_macro_f1': float(row['test_macro_f1']),
            'roc_path': out_dir / 'metrics' / 'test_roc_curve.png',
            'pr_path': out_dir / 'metrics' / 'test_pr_curve.png',
            'confusion_path': out_dir / 'metrics' / 'test_confusion_matrix.png',
            'training_curve_path': out_dir / 'metrics' / 'training_curve.png',
            'source_run': str(run.relative_to(PROJECT_ROOT)),
        })

add_pretrain_run('2_alpanet_pretrain', 'ALPA-Net', 'ALPA-Net pretrain')
add_pretrain_run('4_alpanet_pretrain_no_attention', 'ALPA-Net no attention', 'No-attention pretrain')
add_cv_run('3_alpanet_finetune', 'ALPA-Net transfer CV', '')
add_cv_run('5_alpanet_finetune_no_attention', 'No-attention transfer CV', 'No-attention')
add_single_model_comparison('6_model_pretrain_comparison', 'Baseline pretrain comparison', 'pretrain')
add_single_model_comparison('7_model_finetune_comparison', 'Baseline fine-tune comparison', 'fine-tune')

summary = pd.DataFrame(summary_rows)
if summary.empty:
    raise FileNotFoundError('No model metrics were found under outputs/.')
summary = summary.sort_values('test_macro_f1_mean', ascending=False).reset_index(drop=True)
summary_path = FIG_DIR / 'final_model_comparison_summary.csv'
summary.to_csv(summary_path, index=False)
copy_text_artifact(summary_path)
artifact_df = pd.DataFrame(artifact_rows).sort_values('test_macro_f1', ascending=False).reset_index(drop=True)
artifact_path = FIG_DIR / 'candidate_best_artifacts.csv'
artifact_df.assign(**{c: artifact_df[c].astype(str) for c in ['roc_path', 'pr_path', 'confusion_path', 'training_curve_path']}).to_csv(artifact_path, index=False)
copy_text_artifact(artifact_path)
print(summary[['experiment_group','model','setting','test_macro_f1_mean','test_macro_f1_std','source_run']].to_string(index=False))


             experiment_group                                   model          setting  test_macro_f1_mean  test_macro_f1_std                                              source_run
         ALPA-Net transfer CV                 ALPA-Net full fine-tune    full_finetune            0.859578           0.016481              outputs/3_alpanet_finetune/20260612_150432
     No-attention transfer CV    No-attention ALPA-Net full fine-tune    full_finetune            0.846895           0.015978 outputs/5_alpanet_finetune_no_attention/20260614_051723
Baseline fine-tune comparison                                    LSTM        fine-tune            0.844265                NaN     outputs/7_model_finetune_comparison/20260613_184052
     No-attention transfer CV No-attention ALPA-Net partial fine-tune partial_finetune            0.842969           0.007442 outputs/5_alpanet_finetune_no_attention/20260614_051723
 Baseline pretrain comparison                                   CNN1D         pretrain    

In [38]:

# 5) Final macro-F1 bar chart from the merged summary.
plot_df = summary.copy().sort_values('test_macro_f1_mean', ascending=True)
plot_df['label'] = plot_df['model'] + ' | ' + plot_df['setting']
group_palette = {
    'ALPA-Net pretrain': '#EA580C',
    'ALPA-Net transfer CV': '#2563EB',
    'No-attention pretrain': '#F97316',
    'No-attention transfer CV': '#64748B',
    'Baseline pretrain comparison': '#16A34A',
    'Baseline fine-tune comparison': '#9333EA',
}
colors = [group_palette.get(g, '#475569') for g in plot_df['experiment_group']]
fig_height = max(7.0, 0.40 * len(plot_df) + 1.6)
fig, ax = plt.subplots(figsize=(12.2, fig_height))
ax.barh(plot_df['label'], plot_df['test_macro_f1_mean'], xerr=plot_df['test_macro_f1_std'].fillna(0), color=colors, alpha=0.92, capsize=3)
ax.set_xlim(0, 1.02)
ax.set_xlabel('Test Macro-F1')
ax.set_ylabel('')
ax.set_title('Final Model Comparison on MI Localization')
for y_pos, value in enumerate(plot_df['test_macro_f1_mean']):
    ax.text(min(value + 0.014, 0.99), y_pos, f'{value:.3f}', va='center', fontsize=8.8)
handles = [plt.Line2D([0], [0], color=color, lw=8, label=group) for group, color in group_palette.items() if group in set(plot_df['experiment_group'])]
ax.legend(handles=handles, frameon=False, loc='lower right')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
save_artifact(fig, 'final_macro_f1_model_comparison_bar.png')
print('Saved final_macro_f1_model_comparison_bar.png')


Saved final_macro_f1_model_comparison_bar.png


In [39]:

# 6) Copy best ROC, PR, confusion matrix, and training curve for manuscript.
available = artifact_df[
    artifact_df['roc_path'].map(lambda p: Path(p).exists())
    & artifact_df['pr_path'].map(lambda p: Path(p).exists())
    & artifact_df['confusion_path'].map(lambda p: Path(p).exists())
].copy()
if available.empty:
    raise FileNotFoundError('No candidate model has test ROC, PR, and confusion matrix images.')

best = available.sort_values('test_macro_f1', ascending=False).iloc[0]
copy_artifact(Path(best['roc_path']), 'best_roc_curve.png')
copy_artifact(Path(best['pr_path']), 'best_pr_curve.png')
copy_artifact(Path(best['confusion_path']), 'best_confusion_matrix.png')
if Path(best['training_curve_path']).exists():
    copy_artifact(Path(best['training_curve_path']), 'best_training_curve.png')

metadata = {
    'selection_rule': 'highest test_macro_f1 among candidate models with existing test ROC, PR, and confusion matrix artifacts',
    'overall_best': {
        'model': best['model'],
        'setting': best['setting'],
        'test_macro_f1': float(best['test_macro_f1']),
        'source_run': best['source_run'],
        'roc_path': str(Path(best['roc_path']).relative_to(PROJECT_ROOT)),
        'pr_path': str(Path(best['pr_path']).relative_to(PROJECT_ROOT)),
        'confusion_path': str(Path(best['confusion_path']).relative_to(PROJECT_ROOT)),
        'training_curve_path': str(Path(best['training_curve_path']).relative_to(PROJECT_ROOT)) if Path(best['training_curve_path']).exists() else None,
    },
    'copied_files': {
        'roc': 'fig-tex/best_roc_curve.png',
        'pr': 'fig-tex/best_pr_curve.png',
        'confusion_matrix': 'fig-tex/best_confusion_matrix.png',
        'training_curve': 'fig-tex/best_training_curve.png' if Path(best['training_curve_path']).exists() else None,
    },
}
json_path = FIG_DIR / 'selected_best_fold_figures.json'
json_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
copy_text_artifact(json_path)
print(json.dumps(metadata['overall_best'], indent=2))
print('Saved best ROC, PR, confusion matrix, and metadata.')


{
  "model": " ALPA-Net partial fine-tune",
  "setting": "partial_finetune",
  "test_macro_f1": 0.8900004680495108,
  "source_run": "outputs/3_alpanet_finetune/20260612_150432",
  "roc_path": "outputs/3_alpanet_finetune/20260612_150432/transfer_strategies/partial_finetune/fold_5/metrics/test_roc_curve.png",
  "pr_path": "outputs/3_alpanet_finetune/20260612_150432/transfer_strategies/partial_finetune/fold_5/metrics/test_pr_curve.png",
  "confusion_path": "outputs/3_alpanet_finetune/20260612_150432/transfer_strategies/partial_finetune/fold_5/metrics/test_confusion_matrix.png",
  "training_curve_path": "outputs/3_alpanet_finetune/20260612_150432/transfer_strategies/partial_finetune/fold_5/metrics/training_curve.png"
}
Saved best ROC, PR, confusion matrix, and metadata.


In [40]:

# 7) Final file checklist.
expected_files = [
    'butterworth_bandpass_response.png',
    'data_distribution_ring_chart.png',
    'data_distribution_summary.csv',
    'ecg_raw_vs_butterworth_lead_ii.png',
    'final_macro_f1_model_comparison_bar.png',
    'final_model_comparison_summary.csv',
    'candidate_best_artifacts.csv',
    'best_roc_curve.png',
    'best_pr_curve.png',
    'best_confusion_matrix.png',
    'best_training_curve.png',
    'selected_best_fold_figures.json',
]
for filename in expected_files:
    fig_path = FIG_DIR / filename
    latex_path = LATEX_FIG_DIR / filename
    print(f'{filename}: fig-tex={fig_path.exists()} | file-tex/figs={latex_path.exists()}')


butterworth_bandpass_response.png: fig-tex=True | file-tex/figs=True
data_distribution_ring_chart.png: fig-tex=True | file-tex/figs=True
data_distribution_summary.csv: fig-tex=True | file-tex/figs=True
ecg_raw_vs_butterworth_lead_ii.png: fig-tex=True | file-tex/figs=True
final_macro_f1_model_comparison_bar.png: fig-tex=True | file-tex/figs=True
final_model_comparison_summary.csv: fig-tex=True | file-tex/figs=True
candidate_best_artifacts.csv: fig-tex=True | file-tex/figs=True
best_roc_curve.png: fig-tex=True | file-tex/figs=True
best_pr_curve.png: fig-tex=True | file-tex/figs=True
best_confusion_matrix.png: fig-tex=True | file-tex/figs=True
best_training_curve.png: fig-tex=True | file-tex/figs=True
selected_best_fold_figures.json: fig-tex=True | file-tex/figs=True
